# Magnitude Pruning Pipeline — Full Benchmark

**DETR-ResNet50 + YOLOv5s on COCO val2017**

Pipeline:
1. Baseline: benchmark + COCO eval
2. Global unstructured magnitude pruning at ratios 0.3, 0.5, 0.7, 0.9
3. Benchmark after pruning
4. Recovery fine-tuning (3 epochs, val2017 subset)
5. Benchmark after recovery
6. Export CSV report + plots

**Parameters benchmarked:** params, active_params, sparsity, compression_ratio, model_size_mb, sparse_size_mb, FLOPs(G), latency_ms, fps, mAP50, mAP

In [72]:
!pip install -q ultralytics
!pip install fvcore
print("Dependencies installed.")

Dependencies installed.


In [73]:
import os, sys, copy, time, json, tempfile, warnings, zipfile
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets.utils import download_url
from torchvision.transforms import functional as TF
from PIL import Image

from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

warnings.filterwarnings('ignore')

# ===========================================================================
# SỬA LỖI: MAPPING CHUẨN ĐỂ ĐỒNG BỘ 80 LỚP COCO ID (1-91) VỚI LOCAL INDEX (0-79)
# ===========================================================================
COCO_80_IDS = [
    1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20, 21,
    22, 23, 24, 25, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44,
    46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65,
    67, 70, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 84, 85, 86, 87, 88, 89, 90
]
COCO_TO_YOLO = {coco_id: i for i, coco_id in enumerate(COCO_80_IDS)}

print("All imports OK. COCO Class Mapping initialized.")

All imports OK. COCO Class Mapping initialized.


In [74]:
# ===========================================================================
#  YOLOv5s MODEL WRAPPER (BẢN TỐI ƯU HÓA ĐỒ THỊ PYTORCH THUẦN)
# ===========================================================================
class YOLOv5Model(BaseModel):
    YOLO_IMG_SIZE = 640

    def __init__(self, raw_model_instance=None):
        super().__init__()
        self.num_classes = 80
        self.conf = 0.001
        self.iou = 0.65
        
        if raw_model_instance is not None:
            # Nếu được tạo từ một thực thể PyTorch thuần đã nhân bản
            self.raw_model = raw_model_instance
        else:
            from ultralytics import YOLO
            _hub = YOLO('yolov5s.pt')
            # Lấy chính xác kiến trúc nn.Module thuần của PyTorch để Pruning an toàn
            self.raw_model = _hub.model.model
            
        print(f"YOLOv5 initialized. Internal PyTorch model verified.")

    def train(self, mode: bool = True):
        self.training = mode
        self.raw_model.train(mode)
        return self

    def forward(self, images):
        """Hàm forward phục vụ cho Inference / Evaluation"""
        self.raw_model.eval()
        with torch.no_grad():
            B, _, H, W = images.shape
            x = images * 255.0
            if W != self.YOLO_IMG_SIZE or H != self.YOLO_IMG_SIZE:
                x = F.interpolate(x, size=(self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE),
                                  mode='bilinear', align_corners=False)
            
            raw_out = self.raw_model(x)
            
            if isinstance(raw_out, dict):
                raw_out = raw_out.get('output', raw_out.get('pred', list(raw_out.values())[0]))
            elif isinstance(raw_out, (list, tuple)):
                raw_out = raw_out[0]

            # Khắc phục lỗi đảo ngược ma trận: chuyển [B, 85, 8400] thành [B, 8400, 85]
            if raw_out.dim() == 3 and raw_out.shape[1] < raw_out.shape[2]:
                raw_out = raw_out.transpose(1, 2)

            return self._custom_torchvision_nms(raw_out)

    def forward_train(self, images):
        """Hàm forward phục vụ cho việc tính Loss Gradient khi Training"""
        with torch.enable_grad():
            B, _, H, W = images.shape
            x = images * 255.0
            if W != self.YOLO_IMG_SIZE or H != self.YOLO_IMG_SIZE:
                x = F.interpolate(x, size=(self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE),
                                  mode='bilinear', align_corners=False)
            
            if not x.requires_grad:
                x.requires_grad_(True)
            
            self.raw_model.train()
            raw_out = self.raw_model(x)
            
            if isinstance(raw_out, dict):
                raw_out = raw_out.get('output', raw_out.get('pred', list(raw_out.values())[0]))
            elif isinstance(raw_out, (list, tuple)):
                raw_out = raw_out[0]
            
            if isinstance(raw_out, torch.Tensor):
                # BUG 3a FIX: do NOT detach — only enable grad if truly a no-grad leaf
                if raw_out.requires_grad is False:
                    raw_out.requires_grad_(True)
                
                if raw_out.dim() == 3 and raw_out.shape[1] < raw_out.shape[2]:
                    raw_out = raw_out.transpose(1, 2)
                    
                return raw_out
            else:
                raise TypeError(f"Expected torch.Tensor, got {type(raw_out)}")

    def _custom_torchvision_nms(self, raw_out):
        from torchvision.ops import nms
        B = raw_out.shape[0]
        results = []
        
        for b in range(B):
            pred = raw_out[b] 
            
            # Khắc phục trường hợp mô hình bị lệch số lớp không mong muốn khi clone wrapper
            current_classes = pred.shape[-1] - 5
            
            bboxes = pred[:, :4]
            obj_conf = pred[:, 4]
            class_probs = pred[:, 5:5 + current_classes]
            max_probs, class_ids = class_probs.max(dim=-1)
            scores = max_probs * obj_conf
            
            conf_mask = scores > self.conf
            if not conf_mask.any():
                results.append(torch.zeros((0, 6), device=raw_out.device))
                continue
                
            f_boxes = bboxes[conf_mask]
            f_scores = scores[conf_mask]
            f_class_ids = class_ids[conf_mask]
            
            cx, cy, w, h = f_boxes.unbind(-1)
            x1 = cx - w / 2; y1 = cy - h / 2
            x2 = cx + w / 2; y2 = cy + h / 2
            xyxy_boxes = torch.stack([x1, y1, x2, y2], dim=-1)
            
            keep = nms(xyxy_boxes, f_scores, self.iou)
            
            if len(keep) == 0:
                results.append(torch.zeros((0, 6), device=raw_out.device))
            else:
                final_boxes = xyxy_boxes[keep] / self.YOLO_IMG_SIZE
                final_scores = f_scores[keep].unsqueeze(-1)
                final_classes = f_class_ids[keep].unsqueeze(-1).float()
                
                image_res = torch.cat([final_boxes, final_scores, final_classes], dim=-1)
                results.append(image_res)
                
        return results

    def get_raw_model(self): 
        return self.raw_model
    
    def get_input_size(self): 
        return (3, self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE)


print("YOLOv5Model pure PyTorch deepcopy support enabled.")

YOLOv5Model pure PyTorch deepcopy support enabled.


In [75]:
# ===========================================================================
#  DETR MODEL WRAPPER
# ===========================================================================
class DETRModel(BaseModel):
    DETR_IMG_SIZE = 800

    def __init__(self):
        super().__init__()
        self.model = torch.hub.load('facebookresearch/detr:main', 'detr_resnet50', pretrained=True)
        self.num_classes = self.model.class_embed.out_features - 1

    def forward(self, images, **_):
        # BUG 1 FIX: filter low-confidence background queries before returning
        if images.shape[-1] != self.DETR_IMG_SIZE or images.shape[-2] != self.DETR_IMG_SIZE:
            images = F.interpolate(images, size=(self.DETR_IMG_SIZE, self.DETR_IMG_SIZE),
                                   mode='bilinear', align_corners=False)
        out = self.model(images)
        logits = out['pred_logits']
        boxes = out['pred_boxes']
        prob = F.softmax(logits, dim=-1)
        scores, labels = prob[..., :-1].max(dim=-1)
        cx, cy, w, h = boxes.unbind(-1)
        x1 = (cx - w / 2).clamp(0, 1); y1 = (cy - h / 2).clamp(0, 1)
        x2 = (cx + w / 2).clamp(0, 1); y2 = (cy + h / 2).clamp(0, 1)
        boxes_xyxy = torch.stack([x1, y1, x2, y2], dim=-1)
        results = []
        for b in range(images.shape[0]):
            keep = scores[b] > 0.5  # filter background noise queries
            if not keep.any():
                results.append(torch.zeros((0, 6), device=images.device))
                continue
            dets = torch.stack([
                boxes_xyxy[b, keep, 0], boxes_xyxy[b, keep, 1],
                boxes_xyxy[b, keep, 2], boxes_xyxy[b, keep, 3],
                scores[b, keep], labels[b, keep].float()
            ], dim=-1)
            results.append(dets)
        return results

    def forward_train(self, images):
        B, _, H, W = images.shape
        if W != self.DETR_IMG_SIZE or H != self.DETR_IMG_SIZE:
            images = F.interpolate(images, size=(self.DETR_IMG_SIZE, self.DETR_IMG_SIZE),
                                   mode='bilinear', align_corners=False)
        out = self.model(images)
        return out['pred_logits'], out['pred_boxes']

    def get_raw_model(self): return self.model
    def get_input_size(self): return (3, self.DETR_IMG_SIZE, self.DETR_IMG_SIZE)


print("DETRModel defined.")

DETRModel defined.


In [76]:
# ===========================================================================
#  YOLOv5s MODEL WRAPPER (SỬ DỤNG TORCHVISION NMS - KHÔNG LO LỖI IMPORT)
# ===========================================================================
class YOLOv5Model(BaseModel):
    YOLO_IMG_SIZE = 640

    def __init__(self):
        super().__init__()
        from ultralytics import YOLO
        # Tải mô hình thông qua Ultralytics API để giữ luồng đồ thị chuẩn
        self.model_hub = YOLO('yolov5s.pt')
        self.raw_model = self.model_hub.model
        
        self.conf = 0.001
        self.iou = 0.65
        self.num_classes = 80 

    def train(self, mode: bool = True):
        """Ghi đè cấu hình train/eval để tránh gọi nhầm engine huấn luyện của Ultralytics"""
        self.training = mode
        self.raw_model.train(mode)
        return self

    def forward(self, images):
        """Hàm forward phục vụ cho Inference / Evaluation"""
        self.raw_model.eval()
        with torch.no_grad():
            B, _, H, W = images.shape
            x = images * 255.0
            if W != self.YOLO_IMG_SIZE or H != self.YOLO_IMG_SIZE:
                x = F.interpolate(x, size=(self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE),
                                  mode='bilinear', align_corners=False)
            
            # Chạy qua luồng mạng chuẩn để lấy dự đoán
            raw_out = self.raw_model(x)
            
            if isinstance(raw_out, dict):
                raw_out = raw_out.get('output', raw_out.get('pred', list(raw_out.values())[0]))
            elif isinstance(raw_out, (list, tuple)):
                raw_out = raw_out[0]

            # Hoán vị ma trận: đưa về chuẩn [B, 8400, 85] để bóc tách box/score
            if raw_out.dim() == 3 and raw_out.shape[1] < raw_out.shape[2]:
                raw_out = raw_out.transpose(1, 2)

            return self._custom_torchvision_nms(raw_out)

    def forward_train(self, images):
        """Hàm forward phục vụ cho việc tính Loss Gradient khi Training"""
        with torch.enable_grad():
            B, _, H, W = images.shape
            x = images * 255.0
            if W != self.YOLO_IMG_SIZE or H != self.YOLO_IMG_SIZE:
                x = F.interpolate(x, size=(self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE),
                                  mode='bilinear', align_corners=False)
            
            if not x.requires_grad:
                x.requires_grad_(True)
            
            self.raw_model.train()
            raw_out = self.raw_model(x)
            
            if isinstance(raw_out, dict):
                raw_out = raw_out.get('output', raw_out.get('pred', list(raw_out.values())[0]))
            elif isinstance(raw_out, (list, tuple)):
                raw_out = raw_out[0]
            
            if isinstance(raw_out, torch.Tensor):
                # BUG 3a FIX: do NOT detach — only enable grad if truly a no-grad leaf
                if raw_out.requires_grad is False:
                    raw_out.requires_grad_(True)
                
                if raw_out.dim() == 3 and raw_out.shape[1] < raw_out.shape[2]:
                    raw_out = raw_out.transpose(1, 2)
                    
                return raw_out
            else:
                raise TypeError(f"Expected torch.Tensor, got {type(raw_out)}")

    def _custom_torchvision_nms(self, raw_out):
        """Sử dụng torchvision.ops.nms để loại bỏ hoàn toàn sự phụ thuộc vào Ultralytics"""
        from torchvision.ops import nms
        
        B = raw_out.shape[0]
        results = []
        
        for b in range(B):
            pred = raw_out[b] # Cấu trúc: [8400, 85] (4 box coords, 1 obj_conf, 80 class_probs)
            
            # Trích xuất bounding boxes thô
            bboxes = pred[:, :4]
            
            # Giải mã tọa độ nếu YOLOv5 trả về dạng Anchor-based hoặc phân tích xác suất lớp
            # Thư viện Ultralytics End-to-End trả về trực tiếp tọa độ tâm hoặc biên dạng box
            # Tính toán độ tin cậy kết hợp giữa Objectness và Class Probability tối đa
            obj_conf = pred[:, 4]
            class_probs = pred[:, 5:5 + self.num_classes]
            max_probs, class_ids = class_probs.max(dim=-1)
            scores = max_probs * obj_conf
            
            # Lọc bớt các box có score quá thấp trước khi chạy NMS để tăng tốc
            conf_mask = scores > self.conf
            if not conf_mask.any():
                results.append(torch.zeros((0, 6), device=raw_out.device))
                continue
                
            f_boxes = bboxes[conf_mask]
            f_scores = scores[conf_mask]
            f_class_ids = class_ids[conf_mask]
            
            # Do torchvision NMS nhận vào dạng [x1, y1, x2, y2], ta kiểm tra và chuyển đổi nếu cần
            # YOLO thông thường trả về [cx, cy, w, h] trong hàm thô đầu ra, đưa về xyxy:
            cx, cy, w, h = f_boxes.unbind(-1)
            x1 = cx - w / 2
            y1 = cy - h / 2
            x2 = cx + w / 2
            y2 = cy + h / 2
            xyxy_boxes = torch.stack([x1, y1, x2, y2], dim=-1)
            
            # Kích hoạt hàm NMS độc lập của torchvision
            keep = nms(xyxy_boxes, f_scores, self.iou)
            
            if len(keep) == 0:
                results.append(torch.zeros((0, 6), device=raw_out.device))
            else:
                # Chuẩn hóa tọa độ về khoảng [0, 1] tương ứng với ảnh gốc size 640
                final_boxes = xyxy_boxes[keep] / self.YOLO_IMG_SIZE
                final_scores = f_scores[keep].unsqueeze(-1)
                final_classes = f_class_ids[keep].unsqueeze(-1).float()
                
                # Gom cụm kết quả thành tensor có dạng [x1, y1, x2, y2, score, class_id]
                image_res = torch.cat([final_boxes, final_scores, final_classes], dim=-1)
                results.append(image_res)
                
        return results

    def get_raw_model(self): 
        return self.raw_model
    
    def get_input_size(self): 
        return (3, self.YOLO_IMG_SIZE, self.YOLO_IMG_SIZE)


print("YOLOv5Model is now fully standalone from Ultralytics NMS ops.")

YOLOv5Model is now fully standalone from Ultralytics NMS ops.


In [77]:
class CocoValDataset(Dataset):
    def __init__(self, root, annFile, img_size=640, max_samples=None):
        self.coco = COCO(annFile)
        self.root = root
        self.img_size = img_size
        self.ids = list(self.coco.imgs.keys())
        if max_samples is not None:
            self.ids = self.ids[:max_samples]

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.imgs[img_id]
        fpath = os.path.join(self.root, img_info['file_name'])
        img = Image.open(fpath).convert('RGB')
        orig_w, orig_h = img.size
        img = TF.resize(img, (self.img_size, self.img_size))
        return TF.to_tensor(img), img_id, orig_w, orig_h


class CocoTrainDataset(Dataset):
    def __init__(self, root, annFile, img_size=640, offset=0, max_samples=None):
        self.coco = COCO(annFile)
        self.root = root
        self.img_size = img_size
        self.ids = list(self.coco.imgs.keys())
        if offset > 0: self.ids = self.ids[offset:]
        if max_samples is not None: self.ids = self.ids[:max_samples]

    def __len__(self): return len(self.ids)

    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img_info = self.coco.imgs[img_id]
        fpath = os.path.join(self.root, img_info['file_name'])
        img = Image.open(fpath).convert('RGB')
        orig_w, orig_h = img.size
        img = TF.resize(img, (self.img_size, self.img_size))
        img_tensor = TF.to_tensor(img)
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        boxes = []; labels = []
        for ann in anns:
            if ann.get('ignore', 0) or ann.get('iscrowd', 0): continue
            coco_id = ann['category_id']
            
            # SỬA LỖI: Chỉ thu nạp nhãn thuộc bản đồ map 80 lớp chuẩn nhằm tránh lỗi tràn index (IndexError)
            if coco_id in COCO_TO_YOLO:
                labels.append(COCO_TO_YOLO[coco_id])
            else:
                continue
                
            x, y, w, h = ann['bbox']
            x1 = x / orig_w; y1 = y / orig_h
            x2 = (x + w) / orig_w; y2 = (y + h) / orig_h
            boxes.append([x1, y1, x2, y2])
            
        if not boxes:
            boxes = torch.zeros((0, 4)); labels = torch.zeros((0,), dtype=torch.long)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.long)
        return img_tensor, {'boxes': boxes, 'labels': labels}


print("Datasets redefined successfully.")

Datasets redefined successfully.


In [85]:
# ===========================================================================
#  LOSS FUNCTIONS
# ===========================================================================
def self_bbox_iou(box1, box2):
    inter_x1 = torch.max(box1[..., 0], box2[..., 0])
    inter_y1 = torch.max(box1[..., 1], box2[..., 1])
    inter_x2 = torch.min(box1[..., 2], box2[..., 2])
    inter_y2 = torch.min(box1[..., 3], box2[..., 3])
    inter = (inter_x2 - inter_x1).clamp(0) * (inter_y2 - inter_y1).clamp(0)
    area1 = (box1[..., 2] - box1[..., 0]) * (box1[..., 3] - box1[..., 1])
    area2 = (box2[..., 2] - box2[..., 0]) * (box2[..., 3] - box2[..., 1])
    return inter / (area1 + area2 - inter + 1e-9)


def compute_yolo_loss(pred, targets, num_classes=80):
    device = pred.device
    B = pred.shape[0]
    total_loss = torch.tensor(0.0, device=device, requires_grad=True)

    # ĐỘNG: Xác định số lớp phân loại thực tế dựa trên trục cuối của tensor pred
    # Ví dụ: nếu pred là [B, 8400, 64] -> actual_classes = 64 - 5 = 59
    actual_classes = pred.shape[-1] - 5

    for b in range(B):
        gt_boxes = targets['boxes'][b].to(device)
        gl = targets['labels'][b].to(device)

        if gt_boxes.shape[0] == 0:
            obj_target = torch.zeros(pred.shape[1], device=device)
            obj_loss = F.binary_cross_entropy_with_logits(pred[b, :, 4], obj_target, reduction='sum')
            total_loss = total_loss + obj_loss * 0.1
            continue

        # BUG 3b FIX: gt_boxes are already [x1,y1,x2,y2] normalized — scale directly
        gt_xyxy = gt_boxes * 640.0

        pred_xyxy = pred[b, :, :4]
        
        # Tính toán ma trận IoU nội bộ
        lt = torch.max(pred_xyxy[:, None, :2], gt_xyxy[None, :, :2])
        rb = torch.min(pred_xyxy[:, None, 2:], gt_xyxy[None, :, 2:])
        wh = (rb - lt).clamp(min=0)
        inter = wh[:, :, 0] * wh[:, :, 1]
        area1 = (pred_xyxy[:, 2] - pred_xyxy[:, 0]).clamp(min=0) * (pred_xyxy[:, 3] - pred_xyxy[:, 1]).clamp(min=0)
        area2 = (gt_xyxy[:, 2] - gt_xyxy[:, 0]).clamp(min=0) * (gt_xyxy[:, 3] - gt_xyxy[:, 1]).clamp(min=0)
        iou = inter / (area1[:, None] + area2[None, :] - inter + 1e-6)

        matched_iou, matched_gt_idx = iou.max(dim=1)
        pos_mask = matched_iou > 0.3
        
        box_loss = torch.tensor(0.0, device=device)
        cls_loss = torch.tensor(0.0, device=device)

        if pos_mask.sum() > 0:
            box_loss = F.mse_loss(pred_xyxy[pos_mask], gt_xyxy[matched_gt_idx[pos_mask]], reduction='sum')
            matched_labels = gl[matched_gt_idx[pos_mask]]
            
            # Khống chế giá trị nhãn không vượt quá số lớp mà mô hình có thể dự đoán (Tránh IndexError)
            matched_labels = torch.clamp(matched_labels, max=actual_classes - 1)
            
            # Trích xuất chính xác logits tương ứng với actual_classes
            pred_logits = pred[b, pos_mask, 5:5 + actual_classes]
            
            # Tạo target one-hot khớp hoàn toàn với số lượng kênh của logits
            cls_target = F.one_hot(matched_labels, actual_classes).float()
            cls_loss = F.binary_cross_entropy_with_logits(pred_logits, cls_target, reduction='sum')

        obj_target = torch.zeros(pred.shape[1], device=device)
        obj_target[pos_mask] = matched_iou[pos_mask].detach()
        obj_loss = F.binary_cross_entropy_with_logits(pred[b, :, 4], obj_target, reduction='sum')

        total_loss = total_loss + (box_loss * 0.05 + obj_loss * 1.0 + cls_loss * 0.5)

    return total_loss / B

def compute_detr_loss(pred, targets, num_classes=80):
    """
    BUG 2 FIX: proper Hungarian bipartite matching via scipy.optimize.linear_sum_assignment.
    Old greedy argmin caused assignment conflicts; this guarantees unique 1-to-1 matching.

    pred      : (pred_logits, pred_boxes)
                  pred_logits : (B, 100, num_classes+1)
                  pred_boxes  : (B, 100, 4)  cxcywh normalized [0,1]
    targets   : dict with
                  'boxes'  : list[Tensor(G,4)]  xyxy normalized
                  'labels' : list[LongTensor(G)]
    """
    from scipy.optimize import linear_sum_assignment

    pred_logits, pred_boxes = pred
    device = pred_logits.device
    B = pred_logits.shape[0]
    total_loss = torch.tensor(0.0, device=device)

    for b in range(B):
        gt_boxes_xyxy = targets['boxes'][b].to(device)   # (G, 4) xyxy
        gt_labels     = targets['labels'][b].to(device)  # (G,)
        G = gt_boxes_xyxy.shape[0]

        p_logits = pred_logits[b]   # (100, num_classes+1)
        p_boxes  = pred_boxes[b]    # (100, 4)  cxcywh

        if G == 0:
            bg_target = torch.full((100,), num_classes, dtype=torch.long, device=device)
            loss_ce   = F.cross_entropy(p_logits, bg_target)
            total_loss = total_loss + loss_ce
            continue

        # convert GT xyxy → cxcywh for matching and box loss
        x1g, y1g, x2g, y2g = gt_boxes_xyxy.unbind(-1)
        gt_cxcywh = torch.stack([(x1g+x2g)/2, (y1g+y2g)/2, x2g-x1g, y2g-y1g], dim=-1)

        # cost matrix (detached — index selection only)
        with torch.no_grad():
            l1_cost = torch.cdist(p_boxes, gt_cxcywh, p=1)  # (100, G)

            pcx, pcy, pw, ph = p_boxes.unbind(-1)
            p_x1 = (pcx - pw/2).clamp(0,1); p_y1 = (pcy - ph/2).clamp(0,1)
            p_x2 = (pcx + pw/2).clamp(0,1); p_y2 = (pcy + ph/2).clamp(0,1)
            p_xyxy = torch.stack([p_x1, p_y1, p_x2, p_y2], dim=-1)  # (100,4)

            p_exp = p_xyxy.unsqueeze(1).expand(-1, G, -1)
            g_exp = gt_boxes_xyxy.unsqueeze(0).expand(100, -1, -1)
            ix1 = torch.max(p_exp[...,0], g_exp[...,0]); iy1 = torch.max(p_exp[...,1], g_exp[...,1])
            ix2 = torch.min(p_exp[...,2], g_exp[...,2]); iy2 = torch.min(p_exp[...,3], g_exp[...,3])
            inter = (ix2-ix1).clamp(0) * (iy2-iy1).clamp(0)
            ap = (p_exp[...,2]-p_exp[...,0]).clamp(0) * (p_exp[...,3]-p_exp[...,1]).clamp(0)
            ag = (g_exp[...,2]-g_exp[...,0]).clamp(0) * (g_exp[...,3]-g_exp[...,1]).clamp(0)
            union = ap + ag - inter + 1e-7
            iou   = inter / union
            ex1 = torch.min(p_exp[...,0], g_exp[...,0]); ey1 = torch.min(p_exp[...,1], g_exp[...,1])
            ex2 = torch.max(p_exp[...,2], g_exp[...,2]); ey2 = torch.max(p_exp[...,3], g_exp[...,3])
            encl = (ex2-ex1).clamp(0) * (ey2-ey1).clamp(0) + 1e-7
            giou_cost = 1 - (iou - (encl - union) / encl)
            cost = 2.0 * l1_cost + 2.0 * giou_cost
            row_ind, col_ind = linear_sum_assignment(cost.cpu().numpy())

        row_ind = torch.as_tensor(row_ind, dtype=torch.long, device=device)
        col_ind = torch.as_tensor(col_ind, dtype=torch.long, device=device)

        # classification loss (all 100 queries; unmatched → background class)
        target_classes = torch.full((100,), num_classes, dtype=torch.long, device=device)
        target_classes[row_ind] = gt_labels[col_ind]
        loss_ce = F.cross_entropy(p_logits, target_classes)

        # L1 box loss on matched pairs (cxcywh)
        loss_bbox = F.l1_loss(p_boxes[row_ind], gt_cxcywh[col_ind], reduction='sum') / G

        # GIoU loss on matched pairs (with gradients)
        mp = p_boxes[row_ind]; mg = gt_boxes_xyxy[col_ind]
        mc, mcy, mw, mh = mp.unbind(-1)
        mx1=(mc-mw/2).clamp(0,1); my1=(mcy-mh/2).clamp(0,1)
        mx2=(mc+mw/2).clamp(0,1); my2=(mcy+mh/2).clamp(0,1)
        mp_xy = torch.stack([mx1,my1,mx2,my2], dim=-1)
        ii1=torch.max(mp_xy[:,0],mg[:,0]); ji1=torch.max(mp_xy[:,1],mg[:,1])
        ii2=torch.min(mp_xy[:,2],mg[:,2]); ji2=torch.min(mp_xy[:,3],mg[:,3])
        ia = (ii2-ii1).clamp(0)*(ji2-ji1).clamp(0)
        ua = (mp_xy[:,2]-mp_xy[:,0]).clamp(0)*(mp_xy[:,3]-mp_xy[:,1]).clamp(0)            + (mg[:,2]-mg[:,0]).clamp(0)*(mg[:,3]-mg[:,1]).clamp(0) - ia + 1e-7
        iou_m = ia / ua
        ex1m=torch.min(mp_xy[:,0],mg[:,0]); ey1m=torch.min(mp_xy[:,1],mg[:,1])
        ex2m=torch.max(mp_xy[:,2],mg[:,2]); ey2m=torch.max(mp_xy[:,3],mg[:,3])
        ea = (ex2m-ex1m).clamp(0)*(ey2m-ey1m).clamp(0) + 1e-7
        loss_giou = (1 - (iou_m - (ea - ua) / ea)).sum() / G

        total_loss = total_loss + (loss_ce * 1.0 + loss_bbox * 5.0 + loss_giou * 2.0)

    return total_loss / B

print("Loss functions optimized.")


# --- Smoke tests ---
import torch, torch.nn.functional as F
torch.manual_seed(0)
_B = 2
_pl = torch.randn(_B, 100, 81, requires_grad=True)
_pb = torch.sigmoid(torch.randn(_B, 100, 4)).requires_grad_(True)
_t  = {'boxes': [torch.tensor([[0.1,0.1,0.5,0.5]]), torch.tensor([[0.2,0.2,0.8,0.8]])],
       'labels': [torch.tensor([0]), torch.tensor([1])]}
_ld = compute_detr_loss((_pl, _pb), _t, num_classes=80)
assert torch.isfinite(_ld) and _ld.requires_grad
print(f"[Smoke compute_detr_loss] loss={_ld.item():.4f}  finite=True  requires_grad=True ✓")

_py = torch.randn(2, 8400, 85, requires_grad=True)
_ty = {'boxes': [torch.tensor([[0.1,0.1,0.5,0.5]]), torch.tensor([[0.2,0.2,0.6,0.6]])],
       'labels': [torch.tensor([3]), torch.tensor([7])]}
_ly = compute_yolo_loss(_py, _ty, num_classes=80)
assert torch.isfinite(_ly) and _ly.requires_grad
print(f"[Smoke compute_yolo_loss] loss={_ly.item():.4f}  finite=True  requires_grad=True ✓")


Loss functions optimized.


In [86]:
# ===========================================================================
#  SPARSITY ENFORCEMENT + RECOVERY TRAINING
# ===========================================================================
def enforce_sparsity(model):
    for mod in model.modules():
        if isinstance(mod, (nn.Conv2d, nn.Linear)) and hasattr(mod, 'pruning_mask'):
            mod.weight.data.mul_(mod.pruning_mask)

def recover_model(model, pruner, train_loader, device, epochs=3, lr=1e-4,
                  report_every=50, loss_fn=None, num_classes=80, model_type='yolo'):
    model.train()
    for param in model.parameters():
        param.requires_grad = True
    
    pruner.apply_all_masks()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    
    for epoch in range(epochs):
        running_loss = 0.0
        pbar = tqdm(enumerate(train_loader), total=len(train_loader),
                    desc=f'Epoch {epoch+1}/{epochs}')
        for step, (images, targets) in pbar:
            images = images.to(device)
            
            # Đồng bộ cấu trúc nhãn mục tiêu sang thiết bị xử lý
            if isinstance(targets, dict):
                if 'boxes' in targets and 'labels' in targets:
                    if isinstance(targets['boxes'], list):
                        targets = {
                            'boxes': [b.to(device) for b in targets['boxes']],
                            'labels': [l.to(device) for l in targets['labels']]
                        }
                    else:
                        targets = {
                            'boxes': targets['boxes'].to(device),
                            'labels': targets['labels'].to(device)
                        }

            optimizer.zero_grad()
            pred = model.forward_train(images)
            
            # SỬA LỖI: Gọi chính xác hàm Loss tương ứng với từng kiểu kiến trúc mô hình
            if model_type == 'yolo':
                loss = compute_yolo_loss(pred, targets, num_classes=num_classes)
            else:
                loss = compute_detr_loss(pred, targets, num_classes=num_classes)
            
            loss.backward()
            enforce_sparsity(model)
            optimizer.step()
            enforce_sparsity(model)
            
            running_loss += loss.item()
            if step % report_every == 0 and step > 0:
                pbar.set_postfix({'loss': f'{running_loss/(step+1):.4f}'})
                
    return running_loss / max(len(train_loader), 1)

In [87]:
# ===========================================================================
#  BENCHMARKING HELPERS
# ===========================================================================
def model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix='.pth', delete=False) as f:
        torch.save(model.get_raw_model().state_dict(), f.name)
        sz = os.path.getsize(f.name) / (1024 * 1024)
    os.unlink(f.name)
    return sz


def theoretical_sparse_size_mb(model):
    total_bytes = 0
    for p in model.get_raw_model().parameters():
        total_bytes += (p.data != 0).sum().item() * p.element_size()
    return total_bytes / (1024 * 1024)


def benchmark_model(model, dataloader, device, desc='', num_batches=200):
    model.eval(); model.to(device)
    latencies, total_frames, total_time = [], 0, 0.0
    is_cuda = device.type == 'cuda'
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc=desc,
                                       total=min(num_batches, len(dataloader)))):
            if i >= num_batches: break
            images = batch[0].to(device)
            if images.dim() == 3: images = images.unsqueeze(0)
            if is_cuda: torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = model(images)
            if is_cuda: torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0
            latencies.append(elapsed)
            total_frames += images.size(0); total_time += elapsed
    avg_lat = sum(latencies) / len(latencies) if latencies else 0
    return {'latency_ms': avg_lat * 1000, 'fps': total_frames / max(total_time, 1e-9)}


def compute_flops(model, device='cpu'):
    """Compute FLOPs for the model"""
    try:
        from fvcore.nn import FlopCountAnalysis
        import warnings
        warnings.filterwarnings('ignore', category=UserWarning)
        
        # Get raw model and move to device
        raw_model = model.get_raw_model().to(device).eval()
        
        # Create dummy input
        input_size = (1,) + model.get_input_size()
        dummy = torch.randn(*input_size, device=device)
        
        # For YOLO, we need to handle the forward pass specially
        if isinstance(model, YOLOv5Model):
            # Use the model's forward method directly
            with torch.no_grad():
                flops = FlopCountAnalysis(raw_model, dummy)
            return flops.total() / 1e9
        else:
            # For DETR and others
            with torch.no_grad():
                flops = FlopCountAnalysis(raw_model, dummy)
            return flops.total() / 1e9
    except Exception as e:
        print(f"  [FLOPs] skipped: {e}")
        return float('nan')

def evaluate_coco(model_wrapper, data_loader, coco_gt, device, desc='eval'):
    model_wrapper.eval()
    results = []
    
    # Lấy kích thước ảnh đầu vào thực tế của mô hình (640 đối với YOLO, 800 đối với DETR)
    input_size = model_wrapper.YOLO_IMG_SIZE if hasattr(model_wrapper, 'YOLO_IMG_SIZE') else model_wrapper.DETR_IMG_SIZE
    
    for images, img_ids, orig_ws, orig_hs in tqdm(data_loader, desc=desc):
        images = images.to(device)
        with torch.no_grad():
            outputs = model_wrapper(images)
            
        for b in range(len(outputs)):
            img_id = img_ids[b].item() if isinstance(img_ids, torch.Tensor) else img_ids[b]
            orig_w = orig_ws[b].item() if isinstance(orig_ws, torch.Tensor) else orig_ws[b]
            orig_h = orig_hs[b].item() if isinstance(orig_hs, torch.Tensor) else orig_hs[b]
            dets = outputs[b].cpu().numpy()
            
            if dets.shape[0] == 0: continue
            
            for i in range(dets.shape[0]):
                x1, y1, x2, y2, score, cls_id = dets[i]
                
                # BUG 1 FIX: boxes are normalized [0,1] — scale directly to original size.
                # Old code multiplied by input_size then divided by it (no-op).
                x1_orig = x1 * orig_w
                y1_orig = y1 * orig_h
                x2_orig = x2 * orig_w
                y2_orig = y2 * orig_h
                
                # COCOeval yêu cầu định dạng: [x_min, y_min, width, height]
                w_orig_box = max(0.0, x2_orig - x1_orig)
                h_orig_box = max(0.0, y2_orig - y1_orig)
                
                # Ánh xạ nhãn cục bộ về ID COCO chuẩn (1-91)
                cls_idx = int(cls_id)
                coco_class_id = int(COCO_80_IDS[cls_idx]) if cls_idx < len(COCO_80_IDS) else cls_idx + 1
                
                results.append({
                    'image_id': int(img_id),
                    'category_id': coco_class_id,
                    'bbox': [float(x1_orig), float(y1_orig), float(w_orig_box), float(h_orig_box)],
                    'score': float(score)
                })

    if len(results) == 0:
        print(f"  [WARN] No detections submitted for {desc}!")
        return {'mAP50': 0.0, 'mAP': 0.0, 'precision': 0.0, 'recall': 0.0}
    else:
        print(f"  [DEBUG] {desc}: {len(results)} detections submitted")

    with tempfile.NamedTemporaryFile(mode='w', delete=False, suffix='.json') as f:
        json.dump(results, f)
        res_json_path = f.name

    try:
        coco_dt = coco_gt.loadRes(res_json_path)
        coco_eval = COCOeval(coco_gt, coco_dt, 'bbox')
        coco_eval.params.imgIds = list(data_loader.dataset.ids)
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()
        
        metrics = {
            'mAP50': float(coco_eval.stats[1]),
            'mAP': float(coco_eval.stats[0]),
            'precision': float(coco_eval.stats[4]),
            'recall': float(coco_eval.stats[5])
        }
    except Exception as e:
        print(f"Error during COCO evaluation: {e}")
        metrics = {'mAP50': 0.0, 'mAP': 0.0, 'precision': 0.0, 'recall': 0.0}
    finally:
        if os.path.exists(res_json_path):
            os.remove(res_json_path)

    return metrics

In [88]:
# ===========================================================================
#  COCO VAL2017 DOWNLOAD
# ===========================================================================
DATA_DIR = "/kaggle/working/coco"
os.makedirs(DATA_DIR, exist_ok=True)

VAL_ROOT = os.path.join(DATA_DIR, "val2017")
VAL_ANN  = os.path.join(DATA_DIR, "annotations", "instances_val2017.json")

if not os.path.isdir(VAL_ROOT):
    print("Downloading COCO val2017 images...")
    download_url("http://images.cocodataset.org/zips/val2017.zip",
                 DATA_DIR, "val2017.zip")
    with zipfile.ZipFile(os.path.join(DATA_DIR, "val2017.zip"), "r") as zf:
        zf.extractall(DATA_DIR)
    os.remove(os.path.join(DATA_DIR, "val2017.zip"))

if not os.path.isfile(VAL_ANN):
    print("Downloading COCO annotations...")
    download_url(
        "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
        DATA_DIR, "annotations.zip")
    with zipfile.ZipFile(os.path.join(DATA_DIR, "annotations.zip"), "r") as zf:
        zf.extractall(DATA_DIR)
    os.remove(os.path.join(DATA_DIR, "annotations.zip"))

print(f"COCO val2017 ready: {VAL_ROOT} | {VAL_ANN}")

COCO val2017 ready: /kaggle/working/coco/val2017 | /kaggle/working/coco/annotations/instances_val2017.json


In [89]:
# ===========================================================================
#  LOAD MODELS
# ===========================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

print("Loading DETR-ResNet50...")
detr = DETRModel().to(device).eval()
print(f"  DETR params: {detr.get_params_count():,}")

print("Loading YOLOv5s...")
yolo = YOLOv5Model().to(device).eval()
print(f"  YOLOv5s params: {yolo.get_params_count():,}")

print(f"\nDETR   initial sparsity: {calculate_model_sparsity(detr):.4f}")
print(f"YOLOv5 initial sparsity: {calculate_model_sparsity(yolo):.4f}")

Device: cuda
Loading DETR-ResNet50...


Using cache found in /root/.cache/torch/hub/facebookresearch_detr_main


  DETR params: 41,524,768
Loading YOLOv5s...
PRO TIP 💡 Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.
  YOLOv5s params: 9,153,152

DETR   initial sparsity: 0.0000
YOLOv5 initial sparsity: 0.0000


In [90]:
# ===========================================================================
#  VALIDATION DATALOADERS — separate per model (FIXED: no double-resize)
# ===========================================================================
VAL_MAX_SAMPLES = 500

val_ds_640 = CocoValDataset(VAL_ROOT, VAL_ANN, img_size=640, max_samples=VAL_MAX_SAMPLES)
val_ds_800 = CocoValDataset(VAL_ROOT, VAL_ANN, img_size=800, max_samples=VAL_MAX_SAMPLES)

val_loader_640 = DataLoader(val_ds_640, batch_size=4, shuffle=False, num_workers=2)
val_loader_800 = DataLoader(val_ds_800, batch_size=2, shuffle=False, num_workers=2)

coco_gt = COCO(VAL_ANN)
print(f"Val samples: {len(val_ds_640)}")
print(f"YOLO loader: batch=4, img=640x640")
print(f"DETR loader: batch=2, img=800x800")

loading annotations into memory...
Done (t=0.53s)
creating index...
index created!
loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
loading annotations into memory...
Done (t=0.51s)
creating index...
index created!
Val samples: 500
YOLO loader: batch=4, img=640x640
DETR loader: batch=2, img=800x800


In [91]:
# ===========================================================================
#  MAIN EXPERIMENT LOOP
# ===========================================================================
PRUNING_RATIOS = [0.0, 0.3, 0.5, 0.7, 0.9]
REPORT = []

RECOVERY_OFFSET = VAL_MAX_SAMPLES
RECOVERY_SAMPLES = 200

MODEL_CONFIGS = [
    ("YOLOv5", yolo, val_loader_640),
    ("DETR",   detr, val_loader_800),
    
]

for model_name, model_obj, val_loader in MODEL_CONFIGS:
    print(f"\n{'='*65}")
    print(f"MODEL: {model_name}  |  input: {model_obj.get_input_size()}")
    print(f"{'='*65}")

    baseline_params = model_obj.get_params_count()

    for ratio in PRUNING_RATIOS:
        print(f"\n--- ratio={ratio} ---")

        if ratio == 0.0:
            row = {"model": model_name, "pruning_ratio": ratio, "stage": "baseline"}
            sp = calculate_model_sparsity(model_obj.get_raw_model())
            row["params"] = baseline_params
            row["active_params"] = baseline_params
            row["sparsity"] = sp
            row["compression_ratio"] = 1.0
            row["model_size_mb"] = model_size_mb(model_obj)
            row["sparse_size_mb"] = theoretical_sparse_size_mb(model_obj)
            row["flops_g"] = compute_flops(model_obj, device=str(device))

            bench = benchmark_model(model_obj, val_loader, device,
                                    desc=f"{model_name} baseline bench", num_batches=200)
            row.update(bench)
            metrics = evaluate_coco(model_obj, val_loader, coco_gt, device,
                                     desc=f"{model_name} baseline eval")
            row.update(metrics)
            REPORT.append(row)
            print(f"  Baseline: mAP={metrics['mAP']:.3f}, mAP50={metrics['mAP50']:.3f}, "
                  f"FPS={row['fps']:.1f}, Sparsity={sp*100:.1f}%")
        else:
            cloned = copy.deepcopy(model_obj)
            raw_cloned = cloned.get_raw_model()
            pruner = MagnitudePruner(raw_cloned, pruning_ratio=ratio)
            pruner.prune()
            actual_sparsity = pruner.calculate_sparsity()

            row = {"model": model_name, "pruning_ratio": ratio, "stage": "after_pruning"}
            row["params"] = cloned.get_params_count()
            row["active_params"] = int(row["params"] * (1 - actual_sparsity))
            row["sparsity"] = actual_sparsity
            row["compression_ratio"] = 1.0 / (1.0 - actual_sparsity + 1e-9)
            row["model_size_mb"] = model_size_mb(cloned)
            row["sparse_size_mb"] = theoretical_sparse_size_mb(cloned)
            row["flops_g"] = compute_flops(cloned, device=str(device))

            bench = benchmark_model(cloned, val_loader, device,
                                    desc=f"{model_name} pruned {ratio}", num_batches=200)
            row.update(bench)
            metrics = evaluate_coco(cloned, val_loader, coco_gt, device,
                                     desc=f"{model_name} pruned {ratio} eval")
            row.update(metrics)
            REPORT.append(row)
            print(f"  Pruned ({ratio:.0%}): mAP={metrics['mAP']:.3f}, mAP50={metrics['mAP50']:.3f}, "
                  f"FPS={row['fps']:.1f}, Sparsity={actual_sparsity*100:.1f}%")

            # Recovery fine-tuning
            print(f"  Recovery training on {RECOVERY_SAMPLES} samples...")
            # THÀNH:
            train_loader = DataLoader(
                CocoTrainDataset(
                    VAL_ROOT, VAL_ANN,
                    img_size=model_obj.YOLO_IMG_SIZE if model_name == "YOLOv5" else model_obj.DETR_IMG_SIZE,
                    offset=RECOVERY_OFFSET, max_samples=RECOVERY_SAMPLES
                ), batch_size=2, shuffle=True, num_workers=2,
                collate_fn=lambda batch: (
                    torch.stack([item[0] for item in batch]),
                    {
                        'boxes': [item[1]['boxes'] for item in batch],
                        'labels': [item[1]['labels'] for item in batch]
                    }
                ))

            if model_name == "YOLOv5":
                _ = recover_model(cloned, pruner, train_loader, device,
                                  epochs=3, lr=1e-4, num_classes=cloned.num_classes,
                                  model_type='yolo')
            else:
                _ = recover_model(cloned, pruner, train_loader, device,
                                  epochs=3, lr=1e-4, num_classes=cloned.num_classes,
                                  model_type='detr',
                                  loss_fn=lambda p, t, nc: compute_detr_loss(*p, t, num_classes=nc))

            # after_recovery
            row = {"model": model_name, "pruning_ratio": ratio, "stage": "after_recovery"}
            row["params"] = cloned.get_params_count()
            row["active_params"] = int(row["params"] * (1 - actual_sparsity))
            row["sparsity"] = actual_sparsity
            row["compression_ratio"] = 1.0 / (1.0 - actual_sparsity + 1e-9)
            row["model_size_mb"] = model_size_mb(cloned)
            row["sparse_size_mb"] = theoretical_sparse_size_mb(cloned)
            row["flops_g"] = compute_flops(cloned, device=str(device))

            bench = benchmark_model(cloned, val_loader, device,
                                    desc=f"{model_name} recovered {ratio}", num_batches=200)
            row.update(bench)
            metrics = evaluate_coco(cloned, val_loader, coco_gt, device,
                                     desc=f"{model_name} recovered {ratio} eval")
            row.update(metrics)
            REPORT.append(row)
            print(f"  Recovered ({ratio:.0%}): mAP={metrics['mAP']:.3f}, "
                  f"mAP50={metrics['mAP50']:.3f}, FPS={row['fps']:.1f}")


print(f"\n{'='*65}")
print("EXPERIMENT COMPLETE")
print(f"{'='*65}")


MODEL: YOLOv5  |  input: (3, 640, 640)

--- ratio=0.0 ---


Unsupported operator aten::silu_ encountered 69 time(s)
Unsupported operator aten::add encountered 15 time(s)
Unsupported operator aten::max_pool2d encountered 3 time(s)
Unsupported operator aten::meshgrid encountered 3 time(s)
Unsupported operator aten::mul encountered 4 time(s)
Unsupported operator aten::softmax encountered 1 time(s)
Unsupported operator aten::sub encountered 2 time(s)
Unsupported operator aten::div encountered 1 time(s)
Unsupported operator aten::sigmoid encountered 1 time(s)


YOLOv5 baseline bench:   0%|          | 0/125 [00:00<?, ?it/s]

YOLOv5 baseline eval:   0%|          | 0/125 [00:00<?, ?it/s]

  [DEBUG] YOLOv5 baseline eval: 65939 detections submitted
Loading and preparing results...
DONE (t=0.85s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.59s).
Accumulating evaluation results...
DONE (t=1.01s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average 

...

Epoch 3/3:   0%|          | 0/100 [00:00<?, ?it/s]

Unsupported operator aten::sub encountered 56 time(s)
Unsupported operator aten::pad encountered 2 time(s)
Unsupported operator aten::add encountered 162 time(s)
Unsupported operator aten::rsqrt encountered 53 time(s)
Unsupported operator aten::mul encountered 264 time(s)
Unsupported operator aten::max_pool2d encountered 1 time(s)
Unsupported operator aten::add_ encountered 16 time(s)
Unsupported operator aten::cumsum encountered 2 time(s)
Unsupported operator aten::div encountered 23 time(s)
Unsupported operator aten::pow encountered 1 time(s)
Unsupported operator aten::sin encountered 2 time(s)
Unsupported operator aten::cos encountered 2 time(s)
Unsupported operator aten::repeat encountered 1 time(s)
Unsupported operator aten::masked_fill_ encountered 12 time(s)
Unsupported operator aten::baddbmm encountered 12 time(s)
Unsupported operator aten::softmax encountered 18 time(s)
Unsupported operator aten::sigmoid encountered 1 time(s)
The following submodules of the model were never ca

DETR recovered 0.9:   0%|          | 0/200 [00:00<?, ?it/s]

DETR recovered 0.9 eval:   0%|          | 0/250 [00:00<?, ?it/s]

  [DEBUG] DETR recovered 0.9 eval: 50000 detections submitted
Loading and preparing results...
DONE (t=0.23s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=1.58s).
Accumulating evaluation results...
DONE (t=0.72s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Avera

In [92]:
# ===========================================================================
#  RESULTS TABLE
# ===========================================================================
df = pd.DataFrame(REPORT)
stage_order = {"baseline": 0, "after_pruning": 1, "after_recovery": 2}
df["stage_order"] = df["stage"].map(stage_order)
df = df.sort_values(["model", "stage_order", "pruning_ratio"]).drop(columns="stage_order")

display_cols = ["model", "stage", "pruning_ratio", "params", "active_params",
                "sparsity", "compression_ratio", "model_size_mb", "sparse_size_mb",
                "flops_g", "latency_ms", "fps", "mAP50", "mAP"]
display_cols = [c for c in display_cols if c in df.columns]
print("\n" + df[display_cols].to_string(index=False))

csv_path = "/kaggle/working/magnitude_pruning_report.csv"
df.to_csv(csv_path, index=False)
print(f"\nReport saved to {csv_path}")


 model          stage  pruning_ratio   params  active_params  sparsity  compression_ratio  model_size_mb  sparse_size_mb   flops_g  latency_ms        fps        mAP50          mAP
  DETR       baseline            0.0 41524768       41524768  0.000000           1.000000     158.973125      158.404419 59.564097  104.126656  19.207378 2.757122e-03 1.935403e-03
  DETR  after_pruning            0.3 41524768       29067284  0.300001           1.428574     303.573779      115.034092 59.564097  103.870007  19.254836 2.335896e-03 1.514558e-03
  DETR  after_pruning            0.5 41524768       20762385  0.500000           2.000000     303.573779       86.120853 59.564097  103.554917  19.313424 8.606635e-04 4.764366e-04
  DETR  after_pruning            0.7 41524768       12457374  0.700001           3.333348     303.573779       57.207226 59.564097  103.279130  19.364997 1.394307e-05 2.729008e-06
  DETR  after_pruning            0.9 41524768        4152412  0.900002          10.000155     303.5

In [93]:
# ===========================================================================
#  BENCHMARK SUMMARY: Before vs After Pruning (per ratio)
# ===========================================================================
print("\n\n========== BENCHMARK: BEFORE vs AFTER PRUNING ==========\n")
for model_name in ["DETR", "YOLOv5"]:
    sub = df[df["model"] == model_name]
    baseline = sub[sub["stage"] == "baseline"]
    print(f"\n--- {model_name} ---")
    hdr = f"{'Ratio':<8} {'Stage':<18} {'Params':<12} {'Active':<12} {'Sparsity':<10} "
    hdr += f"{'Compress':<10} {'SizeMB':<8} {'SparseMB':<10} {'FLOPs(G)':<10} "
    hdr += f"{'Lat(ms)':<10} {'FPS':<8} {'mAP50':<8} {'mAP':<8}"
    print(hdr)
    print("-" * 130)

    b = baseline.iloc[0]
    print(f"{'0.0':<8} {'baseline':<18} {b['params']:<12} {b['active_params']:<12} "
          f"{b['sparsity']*100:<9.1f}% {b['compression_ratio']:<10.2f} "
          f"{b['model_size_mb']:<8.1f} {b['sparse_size_mb']:<10.2f} "
          f"{b['flops_g']:<10.2f} {b['latency_ms']:<10.2f} {b['fps']:<8.1f} "
          f"{b['mAP50']:<8.3f} {b['mAP']:<8.3f}")

    for ratio in PRUNING_RATIOS[1:]:
        pruned = sub[(sub["stage"] == "after_pruning") & (sub["pruning_ratio"] == ratio)]
        if pruned.empty: continue
        p = pruned.iloc[0]
        print(f"{ratio:<8} {'after_pruning':<18} {p['params']:<12} {p['active_params']:<12} "
              f"{p['sparsity']*100:<9.1f}% {p['compression_ratio']:<10.2f} "
              f"{p['model_size_mb']:<8.1f} {p['sparse_size_mb']:<10.2f} "
              f"{p['flops_g']:<10.2f} {p['latency_ms']:<10.2f} {p['fps']:<8.1f} "
              f"{p['mAP50']:<8.3f} {p['mAP']:<8.3f}")

        recovered = sub[(sub["stage"] == "after_recovery") & (sub["pruning_ratio"] == ratio)]
        if not recovered.empty:
            r = recovered.iloc[0]
            print(f"{ratio:<8} {'after_recovery':<18} {r['params']:<12} {r['active_params']:<12} "
                  f"{r['sparsity']*100:<9.1f}% {r['compression_ratio']:<10.2f} "
                  f"{r['model_size_mb']:<8.1f} {r['sparse_size_mb']:<10.2f} "
                  f"{r['flops_g']:<10.2f} {r['latency_ms']:<10.2f} {r['fps']:<8.1f} "
                  f"{r['mAP50']:<8.3f} {r['mAP']:<8.3f}")
        print()



========== BENCHMARK: BEFORE vs AFTER PRUNING ==========


--- DETR ---
Ratio    Stage              Params       Active       Sparsity   Compress   SizeMB   SparseMB   FLOPs(G)   Lat(ms)    FPS      mAP50    mAP     
----------------------------------------------------------------------------------------------------------------------------------
0.0      baseline           41524768     41524768     0.0      % 1.00       159.0    158.40     59.56      104.13     19.2     0.003    0.002   
0.3      after_pruning      41524768     29067284     30.0     % 1.43       303.6    115.03     59.56      103.87     19.3     0.002    0.002   
0.3      after_recovery     41524768     29067284     30.0     % 1.43       303.6    115.03     59.56      104.30     19.2     0.000    0.000   

0.5      after_pruning      41524768     20762385     50.0     % 2.00       303.6    86.12      59.56      103.55     19.3     0.001    0.000   
0.5      after_recovery     41524768     20762385     50.0     % 2.00

In [94]:
# ===========================================================================
#  VISUALISATION — 6 subplots: mAP, mAP50, FPS, Latency, Sparse Size, Compression
# ===========================================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Magnitude Pruning: DETR-ResNet50 vs YOLOv5s | COCO val2017", fontsize=13)

COLORS  = {"DETR": "#2196F3", "YOLOv5": "#FF5722"}
METRICS = [
    ("mAP",             "mAP (IoU 0.50:0.95)",  axes[0, 0]),
    ("mAP50",           "mAP50",                 axes[0, 1]),
    ("fps",             "FPS",                   axes[0, 2]),
    ("latency_ms",      "Latency (ms/batch)",    axes[1, 0]),
    ("sparse_size_mb",  "Sparse size (MB)",      axes[1, 1]),
    ("compression_ratio","Compression ratio",    axes[1, 2]),
]

for metric, title, ax in METRICS:
    if metric not in df.columns:
        ax.set_visible(False); continue
    for mname in ["DETR", "YOLOv5"]:
        sub_b = df[(df["model"] == mname) & (df["stage"] == "baseline")]
        sub_p = df[(df["model"] == mname) & (df["stage"] == "after_pruning")]
        sub_r = df[(df["model"] == mname) & (df["stage"] == "after_recovery")]

        xs_p = [0.0] + list(sub_p["pruning_ratio"])
        ys_p = list(sub_b[metric]) + list(sub_p[metric])
        ax.plot(xs_p, ys_p, marker='o', label=f"{mname} pruned", color=COLORS[mname])

        if not sub_r.empty:
            xs_r = [0.0] + list(sub_r["pruning_ratio"])
            ys_r = list(sub_b[metric]) + list(sub_r[metric])
            ax.plot(xs_r, ys_r, marker='s', linestyle='--',
                    label=f"{mname} recovered", color=COLORS[mname], alpha=0.6)

    ax.set_title(title); ax.set_xlabel("Pruning ratio"); ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = "/kaggle/working/magnitude_pruning_plots.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Plot saved to {plot_path}")

Plot saved to /kaggle/working/magnitude_pruning_plots.png


In [95]:
print("\nAll done. Output files:")
print(f"  {csv_path}")
print(f"  {plot_path}")


All done. Output files:
  /kaggle/working/magnitude_pruning_report.csv
  /kaggle/working/magnitude_pruning_plots.png
